# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from tqdm.auto import tqdm

# Import Functions
sys.path.append("../../")
from src.configs.octmnist_config import data_name, data_name_oods, batch_size, eval_batch_size

from src.misc import *
from src.file_manager.filepath import FilePath

from src.models.resnet.model import ResNet18
from src.models.resnet_rue.train import train_resnet_rue
from src.models.resnet_rue.predict import get_rue_predictions_resnet
from src.training.train import train_model_w_best_param

from src.file_manager.load_save_model import load_model
from src.evaluation.inference import get_all_predictions
from src.file_manager.load_save_df import save_pred_df

from src.models.resnet_rue.model import RueResNet18
from src.data_generator.oct_mnist import load_octmnist_data_dict
from src.data_generator.chest_mnist import load_chestmnist_data_dict
from src.data_generator.octdl import load_octdl_data_dict
from src.data_processing.ood_dataset_preprocessing import process_dataset_for_ood, left_join_datasets
from src.data_processing.dataloader import get_pytorch_split_dict_image

from cur_seed import seed
# seed = 2024

fp = FilePath(data_name=data_name, seed=seed)
fp_ood = FilePath(data_name=data_name_oods[0], seed=seed)
fp_ood2 = FilePath(data_name=data_name_oods[1], seed=seed)

# Get Data

In [ ]:
data_dict = load_octmnist_data_dict(fp_preprocessed=fp.get_preprocessed_folder())
data_dict_ood = load_chestmnist_data_dict(fp_preprocessed=fp_ood.get_preprocessed_folder(), only_test=True)
data_dict_ood = process_dataset_for_ood(data_dict, data_dict_ood, seed)
octdl_in_data_dict, octdl_out_data_dict = load_octdl_data_dict(fp_preprocessed=fp_ood2.get_preprocessed_folder())
data_dict = left_join_datasets(data_dict, octdl_in_data_dict)
octdl_in_data_dict = process_dataset_for_ood(data_dict, octdl_in_data_dict, seed)
octdl_out_data_dict = process_dataset_for_ood(data_dict, octdl_out_data_dict, seed)

# Training Param

In [ ]:
params = dict(
    ModelClass=RueResNet18,
    data_dict=data_dict,
    batch_size=batch_size,
    eval_batch_size=batch_size,
    train_model_func=train_resnet_rue,
    metric_to_monitor="rue mae",
    maximise=False,
    train_param_dict = dict(max_epochs=500, lr=0.001, weight_decay=0.001, patience=5, optimizer="adamw"),
    seed=seed,
    fp=fp,
    pytorch_split_dict_func=get_pytorch_split_dict_image
)

# Training 

In [ ]:
resnet_model = load_model(fp=fp, ModelClass=ResNet18, cur_model_name="tuned")
# display_layer_indices(resnet_model)
rue_best_param = {"resnet_model": resnet_model, "last_feat_extractor_layer_index": 0}
rue_model = train_model_w_best_param(
    **params,
    best_param=rue_best_param,
    cur_model_name="tuned"
)

# Prediction

In [ ]:
rue_model = load_model(fp=fp, ModelClass=RueResNet18, cur_model_name="tuned")
pred_df = get_all_predictions(
    model=rue_model, 
    data_dict=data_dict, 
    batch_size=batch_size, 
    eval_batch_size=eval_batch_size,
    pred_func=get_rue_predictions_resnet,
    seed=seed,
    pytorch_split_dict_func=get_pytorch_split_dict_image
) 
save_pred_df(pred_df=pred_df, fp=fp, ModelClass=RueResNet18)

# OOD Prediction

In [ ]:
rue_model = load_model(fp=fp, ModelClass=RueResNet18, cur_model_name="tuned")
ood_dicts = {
    "ood_in_octdl": octdl_in_data_dict, 
    "ood_out_octdl": octdl_out_data_dict,
    "ood_chestmnist": data_dict_ood}
for label, cur_ood_data_dict in tqdm(ood_dicts.items(), total=len(ood_dicts)):
    pred_df_ood = get_all_predictions(
        model=rue_model, 
        data_dict=cur_ood_data_dict, 
        batch_size=batch_size, 
        eval_batch_size=eval_batch_size,
        pred_func=get_rue_predictions_resnet,
        seed=seed,
        pytorch_split_dict_func=get_pytorch_split_dict_image,
        )
    save_pred_df(pred_df=pred_df_ood, fp=fp, ModelClass=RueResNet18, optional_label=label)